# Analyse d'un cube MHD multiphase

Ce notebook reproduit, cellule par cellule, les analyses de CubeAnalysis (hors observations synthétiques) à partir des huit cubes FITS
($n$, $T$, $B_x$, $B_y$, $B_z$, $v_x$, $v_y$, $v_z$). Il n'utilise aucun package maison :
chaque calcul est écrit dans la cellule qui trace la figure.

Renseigne la cellule **Paramètres**, puis exécute tout.

## Paramètres

In [ ]:
FILES = Dict(
    "n"  => "/chemin/vers/density.fits",      # densité [cm^-3]
    "T"  => "/chemin/vers/temperature.fits",  # température [K]
    "Bx" => "/chemin/vers/Bx.fits",           # champ magnétique, unité fixée par B_UNIT_GAUSS
    "By" => "/chemin/vers/By.fits",
    "Bz" => "/chemin/vers/Bz.fits",
    "Vx" => "/chemin/vers/Vx.fits",           # vitesse [km/s]
    "Vy" => "/chemin/vers/Vy.fits",
    "Vz" => "/chemin/vers/Vz.fits",
)

BOX_PC::Float64       = 50.0    # côté de la boîte [pc]
B_UNIT_GAUSS::Float64 = 1e-3    # valeur en gauss d'une unité des cubes B (1e-3 si mG, 1e-6 si μG)
MU::Float64           = 1.27    # masse moléculaire moyenne
GAMMA::Float64        = 5 / 3   # indice adiabatique
PERIODIC::Bool        = true    # conditions aux limites périodiques
STRIDE::Int           = 1       # une cellule sur STRIDE pour les statistiques (1 = toutes)
SAVE::Bool            = true    # enregistre chaque figure en PNG
OUTPUT_DIR::String    = joinpath(pwd(), "figures")

# Phases thermiques [K]
PHASES::Vector{@NamedTuple{name::String, Tmin::Float64, Tmax::Float64}} = [
          (name = "CNM", Tmin = 0.0,    Tmax = 300.0),
          (name = "UNM", Tmin = 300.0,  Tmax = 5000.0),
          (name = "WNM", Tmin = 5000.0, Tmax = Inf)]

# Échelles caractéristiques : injection pour k <= 4 k_min, dissipation sous 4 cellules
INJECTION_MODES::Float64   = 4.0
DISSIPATION_CELLS::Float64 = 4.0

## Packages, style des figures et constantes

In [ ]:
# using Pkg; Pkg.add(["CairoMakie", "LaTeXStrings", "FITSIO", "FFTW"])
using CairoMakie, LaTeXStrings, FITSIO, FFTW
using Statistics, Random, Printf, LinearAlgebra

FFTW.set_num_threads(Threads.nthreads())
mkpath(OUTPUT_DIR)

const PC_CM = 3.0856775814913673e18   # 1 pc en cm
const KB    = 1.380649e-16            # constante de Boltzmann [erg/K]
const MH    = 1.6735575e-24           # masse de l'hydrogène [g]

set_theme!(Theme(fontsize = 18,
    Axis = (xgridvisible = false, ygridvisible = false,
            xticksmirrored = true, yticksmirrored = true,
            xtickalign = 1, ytickalign = 1, xminortickalign = 1, yminortickalign = 1),
    Colorbar = (tickalign = 1, minortickalign = 1),
    Legend = (framevisible = false,)))

# Étiquettes de graduation en LaTeX
function tex_ticks(values)
    map(values) do v
        v == 0 && return L"0"
        e = log10(abs(v))
        if abs(e - round(e)) < 1e-8 && abs(e) >= 3
            return latexstring(v < 0 ? "-" : "", "10^{", round(Int, e), "}")
        end
        return latexstring(string(round(v; sigdigits = 3)))
    end
end
tex_logticks(values) = [abs(log10(v) - round(log10(v))) < 1e-8 ?
    latexstring("10^{", round(Int, log10(v)), "}") :
    latexstring(string(round(v; sigdigits = 2))) for v in values]

# Axe au style commun ; xlog / ylog passent l'axe en log10 avec graduations mineures
function tax(pos; xlog = false, ylog = false, kwargs...)
    opts = Dict{Symbol,Any}(kwargs)
    # des graduations données avec leurs étiquettes (valeurs, noms) n'ont pas besoin de format
    get(opts, :xticks, nothing) isa Tuple || get!(opts, :xtickformat, xlog ? tex_logticks : tex_ticks)
    get(opts, :yticks, nothing) isa Tuple || get!(opts, :ytickformat, ylog ? tex_logticks : tex_ticks)
    if xlog
        opts[:xscale] = log10
        opts[:xminorticks] = IntervalsBetween(9)
        opts[:xminorticksvisible] = true
        opts[:xminorticksmirrored] = true
    end
    if ylog
        opts[:yscale] = log10
        opts[:yminorticks] = IntervalsBetween(9)
        opts[:yminorticksvisible] = true
        opts[:yminorticksmirrored] = true
    end
    return Axis(pos; opts...)
end

texcolorbar(pos, plot; kwargs...) = Colorbar(pos, plot; tickformat = tex_ticks, kwargs...)

# Affiche la figure et l'enregistre si SAVE
function finish(fig, name)
    SAVE && save(joinpath(OUTPUT_DIR, name * ".png"), fig)
    display(fig)
    return nothing
end

COLORS = Makie.wong_colors()

# Affiche des lignes "| a | b |" en tableau HTML (une ligne vide démarre un nouveau tableau)
function show_table(lines)
    io = IOBuffer()
    print(io, "<table>")
    for l in lines
        isempty(strip(l)) && (print(io, "</table><br><table>"); continue)
        occursin(r"^\|[-| ]+\|$", strip(l)) && continue
        cells = strip.(split(strip(l)[2:end-1], "|"))
        print(io, "<tr>", join("<td>" .* cells .* "</td>"), "</tr>")
    end
    print(io, "</table>")
    display(HTML(String(take!(io))))
end

## Lecture des cubes et champs dérivés

Les composantes de $\mathbf{B}$ sont converties en μG. On construit $|B|$, $|v|$ et $P/k_B = nT$.

In [ ]:
read_cube(path) = FITS(path, "r") do f
    Float32.(read(f[1]))
end

for (name, path) in FILES
    isfile(path) || error("Fichier introuvable pour $name : $path")
end

const Cube = Array{Float32,3}
n::Cube  = read_cube(FILES["n"])
T::Cube  = read_cube(FILES["T"])
B_TO_MUG = Float32(B_UNIT_GAUSS * 1e6)
Bx::Cube = read_cube(FILES["Bx"]) .* B_TO_MUG
By::Cube = read_cube(FILES["By"]) .* B_TO_MUG
Bz::Cube = read_cube(FILES["Bz"]) .* B_TO_MUG
Vx::Cube = read_cube(FILES["Vx"])
Vy::Cube = read_cube(FILES["Vy"])
Vz::Cube = read_cube(FILES["Vz"])

shape::NTuple{3,Int} = size(n)
for (name, c) in (("T", T), ("Bx", Bx), ("By", By), ("Bz", Bz), ("Vx", Vx), ("Vy", Vy), ("Vz", Vz))
    size(c) == shape || error("$name a la taille $(size(c)), attendu $shape")
end

Bmag::Cube = @. sqrt(Bx^2 + By^2 + Bz^2)   # μG
Vmag::Cube = @. sqrt(Vx^2 + Vy^2 + Vz^2)   # km/s
Pk::Cube   = n .* T                        # K cm^-3

DX::NTuple{3,Float64} = BOX_PC ./ shape                       # taille d'une cellule par axe [pc]
KMIN::Float64 = 2π / BOX_PC                           # mode fondamental [pc^-1]
KMAX::Float64 = minimum(π .* shape ./ BOX_PC)         # Nyquist [pc^-1]
KINJ::Float64 = INJECTION_MODES * KMIN                # fin de la zone d'injection
KDIS::Float64 = 2KMAX / DISSIPATION_CELLS             # début de la zone de dissipation

# Description de chaque champ : symbole LaTeX, unité, échelle log, palette
FIELDS = [
    (key = "n",    data = n,    tex = "n",       unit = "\\mathrm{cm^{-3}}",       log = true,  cmap = :magma),
    (key = "T",    data = T,    tex = "T",       unit = "\\mathrm{K}",             log = true,  cmap = :inferno),
    (key = "Bmag", data = Bmag, tex = "|B|",     unit = "\\mu\\mathrm{G}",         log = true,  cmap = :plasma),
    (key = "Vmag", data = Vmag, tex = "|v|",     unit = "\\mathrm{km\\,s^{-1}}",   log = true,  cmap = :viridis),
    (key = "Pk",   data = Pk,   tex = "P/k_B",   unit = "\\mathrm{K\\,cm^{-3}}",   log = true,  cmap = :thermal),
    (key = "Bx",   data = Bx,   tex = "B_x",     unit = "\\mu\\mathrm{G}",         log = false, cmap = :balance),
    (key = "By",   data = By,   tex = "B_y",     unit = "\\mu\\mathrm{G}",         log = false, cmap = :balance),
    (key = "Bz",   data = Bz,   tex = "B_z",     unit = "\\mu\\mathrm{G}",         log = false, cmap = :balance),
    (key = "Vx",   data = Vx,   tex = "v_x",     unit = "\\mathrm{km\\,s^{-1}}",   log = false, cmap = :balance),
    (key = "Vy",   data = Vy,   tex = "v_y",     unit = "\\mathrm{km\\,s^{-1}}",   log = false, cmap = :balance),
    (key = "Vz",   data = Vz,   tex = "v_z",     unit = "\\mathrm{km\\,s^{-1}}",   log = false, cmap = :balance),
]
label_lin(f) = latexstring(f.tex, "\\ [", f.unit, "]")
label_log(f) = latexstring("\\log_{10}\\,", f.tex, "\\ [", f.unit, "]")
label_of(f)  = f.log ? label_log(f) : label_lin(f)
tolog(A) = map(x -> x > 0 ? log10(Float64(x)) : NaN, A)

# Résumé
rows = ["| champ | min | médiane | max |", "|---|---|---|---|"]
for f in FIELDS
    v = filter(isfinite, vec(f.data)[1:STRIDE:end])
    push!(rows, @sprintf("| %s | %.3g | %.3g | %.3g |", f.key, minimum(v), median(v), maximum(v)))
end
show_table(rows)
println("Cube $(shape), cellule = $(round(DX[1]; sigdigits = 3)) pc, k_min = $(round(KMIN; sigdigits = 3)) pc^-1, k_Nyq = $(round(KMAX; sigdigits = 3)) pc^-1")

## Outils communs

Histogrammes, ajustement de loi de puissance (moindres carrés pondérés en log-log), choix automatique
du domaine en loi de puissance, et repérage des zones d'injection et de dissipation.

In [ ]:
# Coordonnées des centres de cellules le long de l'axe d [pc]
coords(d) = ((1:shape[d]) .- 0.5) .* DX[d]

# Intervalle de couleur robuste (quantiles 0.5 % et 99.5 %)
function robust_range(A)
    v = filter(isfinite, vec(A))
    isempty(v) && return (0.0, 1.0)
    lo, hi = quantile(v, (0.005, 0.995))
    hi > lo || (hi = lo + max(abs(lo) * 1e-5, 1e-6))
    return (lo, hi)
end

# Échantillon fini (et positif si positive=true) d'un cube, une cellule sur STRIDE
sample(A; positive = false) = [Float64(x) for x in vec(A)[1:STRIDE:end] if isfinite(x) && (!positive || x > 0)]

# Histogramme 1D sur des bords donnés ; la valeur égale au dernier bord tombe dans le dernier bin
function hist1d(values, edges)
    nb = length(edges) - 1
    counts = zeros(nb)
    for v in values
        b = searchsortedlast(edges, v)
        b == nb + 1 && v == edges[end] && (b = nb)
        1 <= b <= nb && (counts[b] += 1)
    end
    centers = (edges[1:end-1] .+ edges[2:end]) ./ 2
    return centers, counts
end

# Pente d'une loi de puissance y ∝ x^α sur [lo, hi], poids optionnels ; renvoie (α, erreur)
function powerlaw_fit(x, y, (lo, hi); w = nothing)
    sel = findall(i -> lo <= x[i] <= hi && isfinite(y[i]) && y[i] > 0 && x[i] > 0, eachindex(x))
    length(sel) >= 3 || return (NaN, NaN)
    X = log10.(Float64.(x[sel])); Y = log10.(Float64.(y[sel]))
    W = w === nothing ? ones(length(sel)) : Float64.(w[sel])
    Xm = sum(W .* X) / sum(W); Ym = sum(W .* Y) / sum(W)
    D = sum(W .* (X .- Xm) .^ 2)
    D > 0 || return (NaN, NaN)
    slope = sum(W .* (X .- Xm) .* (Y .- Ym)) / D
    res = Y .- (Ym .+ slope .* (X .- Xm))
    err = length(sel) > 2 ? sqrt(sum(W .* res .^ 2) / (length(sel) - 2) / D) : NaN
    return slope, err
end

# Plus longue fenêtre en loi de puissance (R² >= 0.95, >= 6 points, >= 0.35 décade) dans [lo, hi]
function best_powerlaw_range(x, y, (lo, hi); min_points = 6, min_decades = 0.35, min_r2 = 0.95)
    c = findall(i -> isfinite(x[i]) && x[i] > 0 && isfinite(y[i]) && y[i] > 0 && lo <= x[i] <= hi, eachindex(x, y))
    length(c) >= min_points || return (lo, hi)
    lx = log10.(Float64.(x[c])); ly = log10.(Float64.(y[c]))
    best = nothing
    for a in 1:length(c)-min_points+1, b in a+min_points-1:length(c)
        lx[b] - lx[a] >= min_decades || continue
        xx = lx[a:b]; yy = ly[a:b]; xm = mean(xx); ym = mean(yy)
        D = sum(abs2, xx .- xm)
        D > 0 || continue
        s = sum((xx .- xm) .* (yy .- ym)) / D
        tot = sum(abs2, yy .- ym)
        r2 = tot > 0 ? 1 - sum(abs2, yy .- (ym .+ s .* (xx .- xm))) / tot : 1.0
        r2 >= min_r2 || continue
        score = (lx[b] - lx[a], r2, b - a + 1)
        (best === nothing || score > best[1]) && (best = (score, a, b))
    end
    best === nothing && return (lo, hi)
    return (x[c[best[2]]], x[c[best[3]]])
end

# Droite de pente donnée ancrée au milieu des points de [lo, hi] (décalage en décades)
function anchored_line(x, y, slope, (lo, hi); offset = 0.0)
    sel = findall(i -> lo <= x[i] <= hi && isfinite(y[i]) && y[i] > 0, eachindex(x))
    length(sel) >= 2 || return (Float64[], Float64[])
    a = sel[cld(length(sel), 2)]
    xs = Float64.(x[sel])
    return xs, y[a] .* 10^offset .* (xs ./ x[a]) .^ slope
end

# Zones d'injection (k <= KINJ) et de dissipation (k >= KDIS) sur un axe en k
function shade_k!(ax)
    vspan!(ax, KMIN, KINJ; color = (:goldenrod, 0.18), label = L"\mathrm{injection}")
    vspan!(ax, KDIS, KMAX; color = (:gray, 0.18), label = L"\mathrm{dissipation}")
    vlines!(ax, [KINJ]; color = :darkorange, linestyle = :dashdot, linewidth = 1.8)
    vlines!(ax, [KDIS]; color = :gray40, linestyle = :dashdot, linewidth = 1.8)
end

# Même chose sur un axe en séparation ℓ = 2π/k [pc]
function shade_l!(ax, lmin, lmax)
    ldis, linj = 2π / KDIS, 2π / KINJ
    lmin < ldis && vspan!(ax, lmin, min(ldis, lmax); color = (:gray, 0.18))
    linj < lmax && vspan!(ax, max(linj, lmin), lmax; color = (:goldenrod, 0.18))
    lmin <= ldis <= lmax && vlines!(ax, [ldis]; color = :gray40, linestyle = :dashdot, linewidth = 1.8)
    lmin <= linj <= lmax && vlines!(ax, [linj]; color = :darkorange, linestyle = :dashdot, linewidth = 1.8)
end

# Indice de phase thermique d'une température (nothing si hors phases)
phase_of(t) = findfirst(p -> p.Tmin <= t < p.Tmax, PHASES)

# Gradient centré : périodique, ou décentré aux bords si periodic = false
# (les globales passent par des arguments par défaut : le corps est compilé pour leurs types concrets)
function gradient3(A, dxs = DX; periodic = PERIODIC)
    gx, gy, gz = similar(A), similar(A), similar(A)
    nx, ny, nz = size(A)
    hx, hy, hz = Float32.(dxs)
    Threads.@threads for k in 1:nz
        for j in 1:ny, i in 1:nx
            if periodic
                gx[i, j, k] = (A[mod1(i + 1, nx), j, k] - A[mod1(i - 1, nx), j, k]) / (2hx)
                gy[i, j, k] = (A[i, mod1(j + 1, ny), k] - A[i, mod1(j - 1, ny), k]) / (2hy)
                gz[i, j, k] = (A[i, j, mod1(k + 1, nz)] - A[i, j, mod1(k - 1, nz)]) / (2hz)
            else
                i1, i2 = max(i - 1, 1), min(i + 1, nx)
                j1, j2 = max(j - 1, 1), min(j + 1, ny)
                k1, k2 = max(k - 1, 1), min(k + 1, nz)
                gx[i, j, k] = (A[i2, j, k] - A[i1, j, k]) / ((i2 - i1) * hx)
                gy[i, j, k] = (A[i, j2, k] - A[i, j1, k]) / ((j2 - j1) * hy)
                gz[i, j, k] = (A[i, j, k2] - A[i, j, k1]) / ((k2 - k1) * hz)
            end
        end
    end
    return gx, gy, gz
end

# Lags logarithmiques (en cellules) jusqu'à min(shape)/divisor
log_lags(divisor, count) = unique(round.(Int, 10 .^ range(0, log10(max(1, minimum(shape) ÷ divisor)); length = count)))

## 1. Coupes

Coupes de chaque champ aux positions 1/4, 1/2 et 3/4 de la boîte, perpendiculairement à $x$, $y$ et $z$.
Échelle de couleur entre les quantiles 0.5 % et 99.5 %.

In [ ]:
SLICE_POSITIONS = (0.25, 0.50, 0.75)
AXIS_NAMES = ("x", "y", "z")

take_slice(A, d, i) = d == 1 ? A[i, :, :] : d == 2 ? A[:, i, :] : A[:, :, i]

for f in FIELDS
    fig = Figure(size = (1350, 360 * length(SLICE_POSITIONS)))
    for (row, p) in enumerate(SLICE_POSITIONS), d in 1:3
        idx = clamp(round(Int, p * (shape[d] - 1)) + 1, 1, shape[d])
        m = take_slice(f.data, d, idx)
        m = f.log ? tolog(m) : Float64.(m)
        u, w = filter(!=(d), 1:3)
        ax = tax(fig[row, d]; aspect = DataAspect(),
            xlabel = latexstring(AXIS_NAMES[u], "\\ [\\mathrm{pc}]"),
            ylabel = latexstring(AXIS_NAMES[w], "\\ [\\mathrm{pc}]"),
            title  = latexstring(AXIS_NAMES[d], " = ", @sprintf("%.1f", coords(d)[idx]), "\\ \\mathrm{pc}"))
        hm = heatmap!(ax, coords(u), coords(w), m; colormap = f.cmap,
                      colorrange = robust_range(m), nan_color = :white)
        d == 3 && texcolorbar(fig[row, 4], hm; label = label_of(f))
    end
    finish(fig, "slices_" * f.key)
end

## 2. Projections

Moyenne et maximum le long de chaque ligne de visée. Pour la densité, on trace la densité de colonne
$N = \sum n\,\Delta l$.

In [ ]:
project(A, d, op) = dropdims(op(Float64.(A); dims = d); dims = d)
column_density(d) = project(n, d, sum) .* (DX[d] * PC_CM)   # cm^-2

for f in FIELDS
    rows = f.key == "n" ?
        [("N", d -> column_density(d), latexstring("\\log_{10}\\,N\\ [\\mathrm{cm^{-2}}]"), true)] :
        [("mean", d -> project(f.data, d, mean),
              f.log ? latexstring("\\log_{10}\\,\\langle ", f.tex, "\\rangle\\ [", f.unit, "]") :
                      latexstring("\\langle ", f.tex, "\\rangle\\ [", f.unit, "]"), f.log),
         ("max", d -> project(f.data, d, maximum),
              f.log ? latexstring("\\log_{10}\\,\\mathrm{max}\\,", f.tex, "\\ [", f.unit, "]") :
                      latexstring("\\mathrm{max}\\,", f.tex, "\\ [", f.unit, "]"), f.log)]
    fig = Figure(size = (1350, 380 * length(rows)))
    for (row, (stat, compute, cblabel, uselog)) in enumerate(rows), d in 1:3
        m = compute(d)
        m = uselog ? tolog(m) : m
        u, w = filter(!=(d), 1:3)
        ax = tax(fig[row, d]; aspect = DataAspect(),
            xlabel = latexstring(AXIS_NAMES[u], "\\ [\\mathrm{pc}]"),
            ylabel = latexstring(AXIS_NAMES[w], "\\ [\\mathrm{pc}]"),
            title  = latexstring("\\mathrm{", stat, "},\\ \\mathrm{LOS}\\ ", AXIS_NAMES[d]))
        hm = heatmap!(ax, coords(u), coords(w), m; colormap = f.cmap,
                      colorrange = robust_range(m), nan_color = :white)
        d == 3 && texcolorbar(fig[row, 4], hm; label = cblabel)
    end
    finish(fig, "projections_" * f.key)
end

## 3. Histogrammes

Pour $n$, $T$, $|B|$, $|v|$ et $P/k_B$ : PDF des valeurs linéaires (coupée au quantile 99.5 % pour que la
queue n'écrase pas tous les bins, mais normalisée sur l'échantillon complet) à côté de la PDF de $\log_{10}$.
Les trois composantes de $\mathbf{B}$, puis de $\mathbf{v}$, sont superposées sur un même graphe.

In [ ]:
NBINS = 80

for f in filter(f -> f.log, FIELDS)
    vals = sample(f.data; positive = true)
    N = length(vals)
    fig = Figure(size = (1300, 540))

    hi = quantile(vals, 0.995)
    edges = collect(range(minimum(vals), hi; length = NBINS + 1))
    c, h = hist1d(vals, edges)
    width = edges[2] - edges[1]
    ax1 = tax(fig[1, 1]; xlabel = label_lin(f), ylabel = L"\mathrm{PDF}")
    barplot!(ax1, c, h ./ (N * width); width = 0.94width, gap = 0, color = (COLORS[1], 0.8))

    lv = log10.(vals)
    edges = collect(range(extrema(lv)...; length = NBINS + 1))
    c, h = hist1d(lv, edges)
    width = edges[2] - edges[1]
    ax2 = tax(fig[1, 2]; xlabel = label_log(f), ylabel = L"\mathrm{PDF}")
    barplot!(ax2, c, h ./ (N * width); width = 0.94width, gap = 0, color = (COLORS[1], 0.8))
    finish(fig, "histogram_" * f.key)
end

for (name, comps, xlab) in (("B", (("B_x", Bx), ("B_y", By), ("B_z", Bz)), L"B_i\ [\mu\mathrm{G}]"),
                            ("v", (("v_x", Vx), ("v_y", Vy), ("v_z", Vz)), L"v_i\ [\mathrm{km\,s^{-1}}]"))
    samples = [sample(c) for (_, c) in comps]
    lo = minimum(minimum, samples); hi = maximum(maximum, samples)
    edges = collect(range(lo, hi; length = NBINS + 1))
    width = edges[2] - edges[1]
    fig = Figure(size = (900, 620))
    ax = tax(fig[1, 1]; xlabel = xlab, ylabel = L"\mathrm{PDF}")
    for (i, ((sym, _), s)) in enumerate(zip(comps, samples))
        c, h = hist1d(s, edges)
        lines!(ax, c, h ./ (length(s) * width); color = COLORS[i], linewidth = 2.6,
               linestyle = (:solid, :dash, :dot)[i], label = latexstring(sym))
    end
    axislegend(ax; position = :rt)
    finish(fig, "histogram_components_" * name)
end

## 4. Diagrammes de phase

Histogrammes 2D de $(\log n, \log T)$ et $(\log n, \log P/k_B)$ en nombre de cellules. On superpose la courbe
d'équilibre thermique (refroidissement de Koyama & Inutsuka 2002, $n_{\mathrm{eq}}(T) = 1/\Lambda(T)$) et les
isothermes 100, 1000 et 8000 K.

In [ ]:
teq_density(t) = 1 / (1e7 * exp(-1.184e5 / (t + 1000)) + 1.4e-2 * sqrt(t) * exp(-92 / t))

function hist2d(x, y, nb)
    xe = collect(range(extrema(x)...; length = nb + 1))
    ye = collect(range(extrema(y)...; length = nb + 1))
    H = zeros(nb, nb)
    for i in eachindex(x, y)
        bx = clamp(floor(Int, (x[i] - xe[1]) / (xe[2] - xe[1])) + 1, 1, nb)
        by = clamp(floor(Int, (y[i] - ye[1]) / (ye[2] - ye[1])) + 1, 1, nb)
        H[bx, by] += 1
    end
    return xe, ye, H
end

ISOTHERMS = [(100.0, :gold), (1000.0, :darkorange), (8000.0, :red)]

for (name, Y, ylab, mode) in (("density_temperature", T,  L"\log_{10}\,T\ [\mathrm{K}]", :T),
                              ("density_pressure",    Pk, L"\log_{10}\,P/k_B\ [\mathrm{K\,cm^{-3}}]", :P))
    keep = [i for i in 1:STRIDE:length(n) if n[i] > 0 && Y[i] > 0 && isfinite(n[i]) && isfinite(Y[i])]
    x = log10.(Float64.(n[keep])); y = log10.(Float64.(Y[keep]))
    xe, ye, H = hist2d(x, y, 160)
    fig = Figure(size = (900, 680))
    ax = tax(fig[1, 1]; xlabel = L"\log_{10}\,n\ [\mathrm{cm^{-3}}]", ylabel = ylab)
    hm = heatmap!(ax, xe, ye, map(c -> c > 0 ? log10(c) : NaN, H); colormap = :magma, nan_color = :white)

    temps = 10 .^ range(log10(5), 6; length = 1200)
    neq = teq_density.(temps)
    ok = (neq .>= 10^xe[1]) .& (neq .<= 10^xe[end])
    yeq = mode == :T ? temps[ok] : neq[ok] .* temps[ok]
    lines!(ax, log10.(neq[ok]), log10.(yeq); color = :royalblue, linewidth = 3,
           label = L"\mathrm{thermal\ equilibrium}")
    nline = 10 .^ range(xe[1], xe[end]; length = 300)
    for (t0, col) in ISOTHERMS
        yy = mode == :T ? fill(log10(t0), 300) : log10.(nline .* t0)
        (maximum(yy) < ye[1] || minimum(yy) > ye[end]) && continue
        lines!(ax, log10.(nline), yy; color = col, linestyle = :dash, linewidth = 2,
               label = latexstring("T = ", @sprintf("%g", t0), "\\ \\mathrm{K}"))
    end
    xlims!(ax, xe[1], xe[end]); ylims!(ax, ye[1], ye[end])
    axislegend(ax; position = :rb)
    texcolorbar(fig[1, 2], hm; label = L"\log_{10}\,N_{\mathrm{cells}}")
    finish(fig, "phase_" * name)
end

## 5. Relations $|B|(n)$ et $|v|(n)$

Moyenne et écart-type de $Y$ dans 30 intervalles logarithmiques de $n$ (entre les quantiles 0.1 % et 99.9 %),
en ne gardant que les intervalles d'au moins 100 cellules.

In [ ]:
function conditional_mean(x, y; bins = 30, minimum_count = 100)
    lo, hi = quantile(filter(>(0), x), (0.001, 0.999))
    edges = 10 .^ range(log10(lo), log10(hi); length = bins + 1)
    cnt = zeros(Int, bins); s1 = zeros(bins); s2 = zeros(bins)
    for i in eachindex(x, y)
        (x[i] < edges[1] || x[i] > edges[end] || !isfinite(y[i])) && continue
        b = clamp(searchsortedlast(edges, x[i]), 1, bins)
        cnt[b] += 1; s1[b] += y[i]; s2[b] += y[i]^2
    end
    keep = cnt .>= minimum_count
    m = s1[keep] ./ cnt[keep]
    sd = sqrt.(max.(s2[keep] ./ cnt[keep] .- m .^ 2, 0))
    return sqrt.(edges[1:end-1] .* edges[2:end])[keep], m, sd
end

for (name, Y, ylab) in (("magnetic_field_vs_density", Bmag, L"|B|\ [\mu\mathrm{G}]"),
                        ("velocity_vs_density",       Vmag, L"|v|\ [\mathrm{km\,s^{-1}}]"))
    idx = [i for i in 1:STRIDE:length(n) if isfinite(n[i]) && n[i] > 0]
    xc, m, sd = conditional_mean(Float64.(n[idx]), Float64.(Y[idx]))
    fig = Figure(size = (900, 620))
    ax = tax(fig[1, 1]; xlog = true, ylog = true, xlabel = L"n\ [\mathrm{cm^{-3}}]", ylabel = ylab)
    band!(ax, xc, max.(m .- sd, 1e-3 .* m), m .+ sd; color = (COLORS[1], 0.18))
    scatterlines!(ax, xc, m; color = COLORS[1], linewidth = 2.6, markercolor = :white,
                  strokecolor = COLORS[1], strokewidth = 1.5, markersize = 8)
    finish(fig, "relation_" * name)
end

## 6. Spectres de puissance

Spectre isotrope $E(k) = \sum_{\mathrm{coquille}} |\hat f(\mathbf{k})|^2 / \Delta k$ en coquilles
logarithmiques (moyenne retirée, transformée normalisée par $\sqrt{N}$). La pente est ajustée dans la plus
longue fenêtre en loi de puissance comprise entre $k_{\mathrm{inj}}$ et $k_{\mathrm{diss}}$, avec un poids
$\sqrt{N_{\mathrm{modes}}}$. Barres d'erreur : $E/\sqrt{N_{\mathrm{modes}}}$. Pour $|v|$, références
Kolmogorov ($k^{-5/3}$) et Burgers ($k^{-2}$).

In [ ]:
# Nombres d'onde des modes d'une rfft [pc^-1]
wavenumbers(sz) = (2π .* rfftfreq(sz[1], sz[1]) ./ BOX_PC,
                   2π .* fftfreq(sz[2], sz[2]) ./ BOX_PC,
                   2π .* fftfreq(sz[3], sz[3]) ./ BOX_PC)

# Transformée de Fourier d'un cube : valeurs non finies remplacées par la moyenne, moyenne retirée
function fourier(A; weight = nothing, weight_power = 0.5)
    a = Float32.(A)
    if weight !== nothing
        for i in eachindex(a)
            w = Float32(weight[i])
            a[i] = isfinite(w) && w >= 0 ? a[i] * w^Float32(weight_power) : NaN32
        end
    end
    good = filter(isfinite, a)
    isempty(good) && error("aucune valeur finie")
    m = mean(good)
    for i in eachindex(a)
        isfinite(a[i]) || (a[i] = m)
    end
    a .-= mean(a)
    return rfft(a) ./ sqrt(Float32(length(a)))
end

# Spectre en coquilles logarithmiques ; renvoie k, E(k) et le nombre de modes
function shell_spectrum(F, sz, kmin, kmax; bins = 80)
    kx, ky, kz = wavenumbers(sz)
    edges = 10 .^ range(log10(kmin), log10(kmax); length = bins + 1)
    S = zeros(bins); M = zeros(Int, bins)
    for k in axes(F, 3), j in axes(F, 2), i in axes(F, 1)
        r = sqrt(kx[i]^2 + ky[j]^2 + kz[k]^2)
        kmin <= r <= kmax || continue
        b = clamp(searchsortedlast(edges, r), 1, bins)
        w = (i == 1 || (iseven(sz[1]) && i == size(F, 1))) ? 1 : 2   # modes conjugués de la rfft
        S[b] += w * abs2(F[i, j, k]); M[b] += w
    end
    keep = M .> 0
    return sqrt.(edges[1:end-1] .* edges[2:end])[keep], (S ./ diff(edges))[keep], M[keep]
end

SPECTRUM_FIELDS = [("n", n, "n"), ("T", T, "T"), ("Bmag", Bmag, "|B|"), ("Vmag", Vmag, "|v|")]

for (key, A, sym) in SPECTRUM_FIELDS
    k, E, M = shell_spectrum(fourier(A), shape, KMIN, KMAX)
    fit_range = best_powerlaw_range(k, E, (KINJ, KDIS))
    slope, err = powerlaw_fit(k, E, fit_range; w = sqrt.(M))
    σ = 1 ./ sqrt.(M)

    fig = Figure(size = (900, 620))
    ax = tax(fig[1, 1]; xlog = true, ylog = true,
             xlabel = L"k\ [\mathrm{pc^{-1}}]", ylabel = latexstring("E_{", sym, "}(k)"))
    shade_k!(ax)
    errorbars!(ax, k, E, E .* σ ./ (1 .+ σ), E .* σ; color = (COLORS[1], 0.5), whiskerwidth = 5)
    scatterlines!(ax, k, E; color = COLORS[1], linewidth = 2.6, markercolor = :white,
                  strokecolor = COLORS[1], strokewidth = 1.2, markersize = 6)
    if isfinite(slope)
        xs, ys = anchored_line(k, E, slope, fit_range)
        lines!(ax, xs, ys; color = :black, linestyle = :dashdot, linewidth = 2.2,
               label = latexstring("\\alpha = ", @sprintf("%.2f", slope), "\\pm", @sprintf("%.2f", err)))
    end
    if key == "Vmag"
        for (i, (s, lab)) in enumerate(((-5 / 3, L"\mathrm{Kolmogorov}\ k^{-5/3}"), (-2.0, L"\mathrm{Burgers}\ k^{-2}")))
            xs, ys = anchored_line(k, E, s, fit_range; offset = 0.2 * (i - 1.5))
            lines!(ax, xs, ys; color = COLORS[i + 1], linestyle = :dash, linewidth = 2, label = lab)
        end
    end
    axislegend(ax; position = :lb)
    finish(fig, "spectrum_" * key)
end

## 7. Spectres vectoriels

Pour un champ vectoriel $\hat{\mathbf{F}}(\mathbf{k})$, la partie compressive est
$|\mathbf{k}\cdot\hat{\mathbf{F}}|^2/k^2$ et la partie solénoïdale le reste. On trace $\mathbf{B}$, $\mathbf{v}$,
la vorticité $\boldsymbol{\omega} = \nabla\times\mathbf{v}$ (calculée en Fourier, $i\mathbf{k}\times\hat{\mathbf{v}}$)
et $\sqrt{n}\,\mathbf{v}$ (spectre d'énergie cinétique).

In [ ]:
function vector_spectrum(cx, cy, cz; weight = nothing, curl = false, bins = 80)
    Fx, Fy, Fz = (fourier(c; weight = weight) for c in (cx, cy, cz))
    kx, ky, kz = wavenumbers(shape)
    kmin, kmax = KMIN, KMAX
    edges = 10 .^ range(log10(kmin), log10(kmax); length = bins + 1)
    tot = zeros(bins); sol = zeros(bins); com = zeros(bins); M = zeros(Int, bins)
    for k in axes(Fx, 3), j in axes(Fx, 2), i in axes(Fx, 1)
        k1, k2, k3 = kx[i], ky[j], kz[k]
        kk = k1^2 + k2^2 + k3^2
        kmin^2 <= kk <= kmax^2 || continue
        b = clamp(searchsortedlast(edges, sqrt(kk)), 1, bins)
        w = (i == 1 || (iseven(shape[1]) && i == size(Fx, 1))) ? 1 : 2
        a1, a2, a3 = Fx[i, j, k], Fy[i, j, k], Fz[i, j, k]
        if curl
            a1, a2, a3 = im * (k2 * a3 - k3 * a2), im * (k3 * a1 - k1 * a3), im * (k1 * a2 - k2 * a1)
        end
        e = abs2(a1) + abs2(a2) + abs2(a3)
        l = abs2(k1 * a1 + k2 * a2 + k3 * a3) / kk
        tot[b] += w * e; com[b] += w * l; sol[b] += w * max(e - l, 0); M[b] += w
    end
    keep = M .> 0
    dk = diff(edges)
    kc = sqrt.(edges[1:end-1] .* edges[2:end])
    return kc[keep], (tot ./ dk)[keep], (sol ./ dk)[keep], (com ./ dk)[keep], M[keep]
end

VECTOR_SPECTRA = [
    ("magnetic",  (Bx, By, Bz), nothing, false, "E_B(k)",         false),
    ("velocity",  (Vx, Vy, Vz), nothing, false, "E_v(k)",         true),
    ("vorticity", (Vx, Vy, Vz), nothing, true,  "E_{\\omega}(k)", false),
    ("kinetic",   (Vx, Vy, Vz), n,       false, "E_{\\sqrt{n}v}(k)", false),
]

for (name, comps, weight, curl, ylab, references) in VECTOR_SPECTRA
    k, Etot, Esol, Ecom, M = vector_spectrum(comps...; weight = weight, curl = curl)
    fig = Figure(size = (900, 620))
    ax = tax(fig[1, 1]; xlog = true, ylog = true, xlabel = L"k\ [\mathrm{pc^{-1}}]", ylabel = latexstring(ylab))
    shade_k!(ax)
    σ = 1 ./ sqrt.(M)
    errorbars!(ax, k, Etot, Etot .* σ ./ (1 .+ σ), Etot .* σ; color = (:black, 0.4), whiskerwidth = 5)
    lines!(ax, k, Etot; color = :black, linewidth = 2.8, label = L"\mathrm{total}")
    if !curl
        good = Esol .> 0
        lines!(ax, k[good], Esol[good]; color = COLORS[1], linewidth = 2.4, label = L"\mathrm{solenoidal}")
        good = Ecom .> 0
        lines!(ax, k[good], Ecom[good]; color = COLORS[2], linewidth = 2.4, linestyle = :dash, label = L"\mathrm{compressive}")
    end
    if references
        for (i, (s, lab)) in enumerate(((-5 / 3, L"\mathrm{Kolmogorov}\ k^{-5/3}"), (-2.0, L"\mathrm{Burgers}\ k^{-2}")))
            xs, ys = anchored_line(k, Etot, s, (KINJ, KDIS); offset = 0.2 * (i - 1.5))
            lines!(ax, xs, ys; color = COLORS[i + 2], linestyle = :dash, linewidth = 2, label = lab)
        end
    end
    axislegend(ax; position = :lb)
    finish(fig, "vector_spectrum_" * name)
end

## 8. Cohérence entre $n$ et $|B|$

$\mathcal{C}(k) = |\sum \hat n \hat B^*|^2 / (\sum |\hat n|^2 \sum |\hat B|^2)$ par coquille : 1 si les deux champs
sont parfaitement corrélés à cette échelle, 0 s'ils sont indépendants.

In [ ]:
function coherence(A1, A2; bins = 80)
    F1, F2 = fourier(A1), fourier(A2)
    kx, ky, kz = wavenumbers(shape)
    kmin, kmax = KMIN, KMAX
    edges = 10 .^ range(log10(kmin), log10(kmax); length = bins + 1)
    C = zeros(ComplexF64, bins); P1 = zeros(bins); P2 = zeros(bins); M = zeros(Int, bins)
    for k in axes(F1, 3), j in axes(F1, 2), i in axes(F1, 1)
        r = sqrt(kx[i]^2 + ky[j]^2 + kz[k]^2)
        kmin <= r <= kmax || continue
        b = clamp(searchsortedlast(edges, r), 1, bins)
        w = (i == 1 || (iseven(shape[1]) && i == size(F1, 1))) ? 1 : 2
        a, c = F1[i, j, k], F2[i, j, k]
        C[b] += w * a * conj(c); P1[b] += w * abs2(a); P2[b] += w * abs2(c); M[b] += w
    end
    keep = M .> 0
    return sqrt.(edges[1:end-1] .* edges[2:end])[keep], (abs2.(C) ./ max.(P1 .* P2, eps()))[keep]
end

k, coh = coherence(n, Bmag)
fig = Figure(size = (900, 620))
ax = tax(fig[1, 1]; xlog = true, xlabel = L"k\ [\mathrm{pc^{-1}}]", ylabel = L"\mathcal{C}_{n,|B|}(k)")
shade_k!(ax)
lines!(ax, k, coh; color = COLORS[1], linewidth = 2.8)
ylims!(ax, 0, 1.05)
axislegend(ax; position = :rt)
finish(fig, "cross_spectrum_density_Bmag")

## 9. Alignement de $\nabla n$ avec $\mathbf{B}$ (HRO)

Dans chaque cellule, $|\cos\phi| = |\nabla n\cdot\mathbf{B}|/(|\nabla n||\mathbf{B}|)$, pondéré par $|\nabla n|$.
Gauche : distribution de $|\cos\phi|$ dans 12 intervalles logarithmiques de densité (normalisée par ligne).
Droite : $\zeta = (A_\parallel - A_\perp)/(A_\parallel + A_\perp)$ avec $A_\parallel$ le poids des cellules à
$\phi < 22.5^\circ$ et $A_\perp$ à $\phi > 67.5^\circ$ ; $\zeta > 0$ : gradients parallèles à $\mathbf{B}$
(structures perpendiculaires au champ). Erreurs par bootstrap (200 tirages sur au plus 20 000 cellules par
intervalle, échantillonnées par réservoir).

In [ ]:
function hro(gx, gy, gz, bx, by, bz, cond; nbins = 12, nangles = 30, minimum_count = 100,
             replicates = 200, max_samples = 20_000, seed = 1234)
    lo, hi = quantile(filter(x -> isfinite(x) && x > 0, vec(cond)[1:max(1, length(cond) ÷ 1_000_000):end]), (0.001, 0.999))
    edges = 10 .^ range(log10(lo), log10(hi); length = nbins + 1)
    H = zeros(nbins, nangles); cnt = zeros(Int, nbins)
    par = zeros(nbins); perp = zeros(nbins)
    cpar, cperp = cosd(22.5), cosd(67.5)
    rng = MersenneTwister(seed)
    reservoir = [Tuple{Float32,Float32}[] for _ in 1:nbins]
    for i in eachindex(cond)
        c = cond[i]
        (isfinite(c) && edges[1] <= c <= edges[end]) || continue
        g = sqrt(gx[i]^2 + gy[i]^2 + gz[i]^2); bm = sqrt(bx[i]^2 + by[i]^2 + bz[i]^2)
        (g > 0 && bm > 0) || continue
        cosφ = clamp(abs(gx[i] * bx[i] + gy[i] * by[i] + gz[i] * bz[i]) / (g * bm), 0, 1)
        b = clamp(searchsortedlast(edges, c), 1, nbins)
        a = clamp(floor(Int, cosφ * nangles) + 1, 1, nangles)
        H[b, a] += g; cnt[b] += 1
        cosφ >= cpar && (par[b] += g)
        cosφ <= cperp && (perp[b] += g)
        r = reservoir[b]
        if length(r) < max_samples
            push!(r, (Float32(cosφ), Float32(g)))
        else
            j = rand(rng, 1:cnt[b])
            j <= max_samples && (r[j] = (Float32(cosφ), Float32(g)))
        end
    end
    ζ = fill(NaN, nbins); ζerr = fill(NaN, nbins)
    for b in 1:nbins
        s = sum(H[b, :]); s > 0 && (H[b, :] ./= s)
        (cnt[b] >= minimum_count && par[b] + perp[b] > 0) || continue
        ζ[b] = (par[b] - perp[b]) / (par[b] + perp[b])
        r = reservoir[b]
        boot = Float64[]
        for _ in 1:replicates
            p = 0.0; q = 0.0
            for _ in eachindex(r)
                cφ, w = r[rand(rng, eachindex(r))]
                cφ >= cpar && (p += w)
                cφ <= cperp && (q += w)
            end
            p + q > 0 && push!(boot, (p - q) / (p + q))
        end
        length(boot) > 1 && (ζerr[b] = std(boot))
    end
    centers = sqrt.(edges[1:end-1] .* edges[2:end])
    angles = collect(range(0.5 / nangles, 1 - 0.5 / nangles; length = nangles))
    return centers, angles, H, ζ, ζerr
end

gx, gy, gz = gradient3(n)
nc, angles, H, ζ, ζerr = hro(gx, gy, gz, Bx, By, Bz, n)

fig = Figure(size = (1200, 560))
ax1 = tax(fig[1, 1]; ylog = true, xlabel = L"|\cos\phi|", ylabel = L"n\ [\mathrm{cm^{-3}}]")
hm = heatmap!(ax1, angles, nc, permutedims(H); colormap = :viridis)
texcolorbar(fig[1, 2], hm; label = L"\mathrm{normalized}\ |\nabla n|\ \mathrm{weight}")
ax2 = tax(fig[1, 3]; xlog = true, xlabel = L"n\ [\mathrm{cm^{-3}}]", ylabel = L"\zeta")
hlines!(ax2, [0.0]; color = :gray50, linestyle = :dash)
ok = isfinite.(ζ)
errorbars!(ax2, nc[ok], ζ[ok], replace(ζerr[ok], NaN => 0.0); color = (COLORS[4], 0.7), whiskerwidth = 7)
scatterlines!(ax2, nc[ok], ζ[ok]; color = COLORS[4], linewidth = 2.6, markercolor = :white,
              strokecolor = COLORS[4], strokewidth = 1.5, markersize = 8)
ylims!(ax2, -1.05, 1.05)
finish(fig, "alignment_density_gradient_vs_B")
gx = gy = gz = nothing; GC.gc()

### Paramètre $\xi$ (Ibáñez-Mejía et al.)

$\xi = (N_\parallel - N_\perp)/(N_\parallel + N_\perp)$ en nombre de cellules, avec $|\cos| \ge 0.75$ (parallèle)
et $|\cos| \le 0.25$ (perpendiculaire), dans 25 intervalles logarithmiques de $n \in [10^{-2}, 10^3]$
(au moins 1000 cellules). Deux alignements : $\nabla n$ avec $\mathbf{B}$ (gradient non périodique) et
$\delta\mathbf{v} = \mathbf{v} - \langle\mathbf{v}\rangle_n$ avec $\mathbf{B}$. Erreur : le maximum entre une erreur
analytique tirée de la distribution des cosinus et la dispersion de $\xi$ sur $4^3$ sous-blocs.

In [ ]:
function xi_curves(n, gx, gy, gz, bx, by, bz, vx, vy, vz; nbins = 25, nrange = (1e-2, 1e3),
                   minimum_count = 1000, blocks = 4, cpar = 0.75, cperp = 0.25, ncos = 40)
    edges = 10 .^ range(log10(nrange[1]), log10(nrange[2]); length = nbins + 1)
    cos_edges = collect(range(0, 1; length = ncos + 1))
    par_bins  = [i for i in 1:ncos if cos_edges[i] >= cpar]
    perp_bins = [i for i in 1:ncos if cos_edges[i+1] <= cperp]
    sz = size(n); bsize = cld.(sz, blocks); nblk = blocks^3

    # vitesse moyenne pondérée par la densité
    w = 0.0; mv = zeros(3)
    for i in eachindex(n)
        (isfinite(n[i]) && n[i] > 0) || continue
        w += n[i]; mv .+= n[i] .* (vx[i], vy[i], vz[i])
    end
    mv ./= w

    curves = map(1:2) do which   # 1 : ∇n·B, 2 : δv·B
        counts = zeros(nbins, 2); bcounts = zeros(nbins, nblk, 2); bsamples = zeros(Int, nbins, nblk)
        hist = zeros(nbins, ncos); samples = zeros(Int, nbins)
        for I in CartesianIndices(n)
            c = n[I]
            (isfinite(c) && nrange[1] <= c < nrange[2]) || continue
            b = clamp(searchsortedlast(edges, c), 1, nbins)
            B = (bx[I], by[I], bz[I])
            a = which == 1 ? (gx[I], gy[I], gz[I]) : (vx[I] - mv[1], vy[I] - mv[2], vz[I] - mv[3])
            na = sqrt(sum(abs2, a)); nb = sqrt(sum(abs2, B))
            (na > 0 && nb > 0) || continue
            cs = clamp(abs(sum(a .* B)) / (na * nb), 0, 1)
            blk = ntuple(d -> clamp(fld(I[d] - 1, bsize[d]) + 1, 1, blocks), 3)
            blk_index = blk[1] + blocks * (blk[2] - 1) + blocks^2 * (blk[3] - 1)
            samples[b] += 1; bsamples[b, blk_index] += 1
            hist[b, clamp(searchsortedlast(cos_edges, cs), 1, ncos)] += 1
            cat = cs >= cpar ? 1 : cs <= cperp ? 2 : 0
            if cat > 0
                counts[b, cat] += 1; bcounts[b, blk_index, cat] += 1
            end
        end
        xc = Float64[]; ξ = Float64[]; err = Float64[]
        for b in 1:nbins
            (samples[b] >= minimum_count && counts[b, 1] + counts[b, 2] > 0) || continue
            h = hist[b, :] ./ sum(hist[b, :])
            P = sum(h[par_bins]); Q = sum(h[perp_bins])
            sP = length(par_bins) > 1 ? std(h[par_bins]) : 0.0
            sQ = length(perp_bins) > 1 ? std(h[perp_bins]) : 0.0
            analytic = P + Q > 0 ? sqrt(4 * (Q^2 * sP^2 + P^2 * sQ^2) / (P + Q)^4) : 0.0
            bξ = [(bcounts[b, k, 1] - bcounts[b, k, 2]) / (bcounts[b, k, 1] + bcounts[b, k, 2])
                  for k in 1:nblk if bsamples[b, k] >= 20 && bcounts[b, k, 1] + bcounts[b, k, 2] > 0]
            push!(xc, sqrt(edges[b] * edges[b+1]))
            push!(ξ, (counts[b, 1] - counts[b, 2]) / (counts[b, 1] + counts[b, 2]))
            push!(err, length(bξ) >= 3 ? max(analytic, std(bξ)) : analytic)
        end
        (xc, ξ, err)
    end
    return curves
end

gx, gy, gz = gradient3(n; periodic = false)
(xg, ξg, eg), (xv, ξv, ev) = xi_curves(n, gx, gy, gz, Bx, By, Bz, Vx, Vy, Vz)
gx = gy = gz = nothing; GC.gc()

fig = Figure(size = (1200, 500))
allx = vcat(xg, xv)
xl = isempty(allx) ? (1e-2, 1e3) : (10.0^floor(log10(minimum(allx))), 10.0^ceil(log10(maximum(allx))))
for (col, (x, ξ, e, lab)) in enumerate(((xg, ξg, eg, L"\xi_{\nabla n,B}"), (xv, ξv, ev, L"\xi_{\delta v,B}")))
    ax = tax(fig[1, col]; xlog = true, xlabel = L"n\ [\mathrm{cm^{-3}}]", ylabel = lab)
    band!(ax, collect(xl), [0.0, 0.0], [1.0, 1.0]; color = (COLORS[1], 0.12))
    band!(ax, collect(xl), [-1.0, -1.0], [0.0, 0.0]; color = (COLORS[2], 0.12))
    hlines!(ax, [0.0]; color = :gray50, linestyle = :dash)
    if !isempty(x)
        errorbars!(ax, x, ξ, e; color = (:black, 0.7), whiskerwidth = 8)
        scatterlines!(ax, x, ξ; color = :black, linewidth = 2.6, markercolor = :white,
                      strokecolor = :black, strokewidth = 1.5, markersize = 9)
    end
    xlims!(ax, xl...); ylims!(ax, -1, 1)
end
finish(fig, "xi_density_gradient_vs_B")

## 10. Table des grandeurs physiques

Par cellule : $\mathcal{M}_s = |v|/c_s$ avec $c_s = \sqrt{\gamma k_B T/(\mu m_H)}$,
$\mathcal{M}_A = |v|/v_A$ avec $v_A = |B|/\sqrt{4\pi\mu m_H n}$, et $\beta = 8\pi n k_B T/|B|^2$.
On donne la médiane et l'intervalle $[q_{16}, q_{84}]$ pondérés en volume puis en masse ($\propto n$).
Pour $\mathbf{B}$ et $\mathbf{v}$ : $|\langle\mathbf{F}\rangle|$, $\delta F_{\mathrm{rms}} = \sqrt{\langle|\mathbf{F}|^2\rangle - |\langle\mathbf{F}\rangle|^2}$ et leur rapport.

In [ ]:
function weighted_quantiles(v, w, ps)
    o = sortperm(v); cw = cumsum(w[o]); tot = cw[end]
    return [v[o[min(searchsortedfirst(cw, p * tot), length(o))]] for p in ps]
end

function physics_per_cell(n, T, bx, by, bz, vx, vy, vz; stride = STRIDE)
    Ms = Float64[]; wMs = Float64[]; MA = Float64[]; β = Float64[]; wB = Float64[]
    for i in 1:stride:length(n)
        (isfinite(n[i]) && n[i] > 0 && isfinite(T[i]) && T[i] > 0) || continue
        v = 1e5 * sqrt(vx[i]^2 + vy[i]^2 + vz[i]^2)                 # cm/s
        isfinite(v) || continue
        cs = sqrt(GAMMA * KB * T[i] / (MU * MH))
        push!(Ms, v / cs); push!(wMs, n[i])
        B = 1e-6 * sqrt(bx[i]^2 + by[i]^2 + bz[i]^2)                 # G
        (isfinite(B) && B > 0) || continue
        push!(MA, v / (B / sqrt(4π * MU * MH * n[i])))
        push!(β, 8π * n[i] * KB * T[i] / B^2)
        push!(wB, n[i])
    end
    return Ms, wMs, MA, β, wB
end

# |⟨F⟩|, δF_rms et leur rapport, pondérés par w (nothing = volume)
function fluctuation(cx, cy, cz; w = nothing, stride = STRIDE)
    s = zeros(3); s2 = 0.0; W = 0.0
    for i in 1:stride:length(cx)
        a = (cx[i], cy[i], cz[i]); all(isfinite, a) || continue
        wi = w === nothing ? 1.0 : (isfinite(w[i]) && w[i] > 0 ? Float64(w[i]) : 0.0)
        s .+= wi .* a; s2 += wi * sum(abs2, a); W += wi
    end
    m = sqrt(sum(abs2, s ./ W)); δ = sqrt(max(s2 / W - m^2, 0))
    return m, δ, δ / m
end

Ms, wMs, MA, β, wB = physics_per_cell(n, T, Bx, By, Bz, Vx, Vy, Vz)
fmt(x) = isfinite(x) ? @sprintf("%.3g", x) : "n/a"
stat_row(name, v, w) = begin
    qv = quantile(v, (0.16, 0.5, 0.84)); qm = weighted_quantiles(v, w, (0.16, 0.5, 0.84))
    "| $name | $(fmt(qv[2])) | [$(fmt(qv[1])), $(fmt(qv[3]))] | $(fmt(qm[2])) | [$(fmt(qm[1])), $(fmt(qm[3]))] |"
end
lines_md = ["| | médiane (volume) | [q16, q84] (volume) | médiane (masse) | [q16, q84] (masse) |",
            "|---|---|---|---|---|",
            stat_row("Mach sonique", Ms, wMs), stat_row("Mach Alfvénique", MA, wB), stat_row("β plasma", β, wB)]
push!(lines_md, "", "| | volume | masse |", "|---|---|---|")
for (name, comps, unit) in (("B", (Bx, By, Bz), "μG"), ("v", (Vx, Vy, Vz), "km/s"))
    fv = fluctuation(comps...); fm = fluctuation(comps...; w = n)
    push!(lines_md, "| ‖⟨$name⟩‖ [$unit] | $(fmt(fv[1])) | $(fmt(fm[1])) |",
                    "| δ$(name)_rms [$unit] | $(fmt(fv[2])) | $(fmt(fm[2])) |",
                    "| δ$(name)_rms / ‖⟨$name⟩‖ | $(fmt(fv[3])) | $(fmt(fm[3])) |")
end
show_table(lines_md)

## 11. PDF de la densité

$s = \ln(n/\langle n\rangle)$. On compare l'histogramme à la lognormale contrainte
($\langle s\rangle = -\sigma_s^2/2$) et à une gaussienne libre. Paramètre de forçage déduit de
$\sigma_s^2 = \ln(1 + b^2\mathcal{M}_s^2)$, et sa version magnétisée
$\sigma_s^2 = \ln(1 + b^2\mathcal{M}_s^2\beta/(\beta+1))$, avec $\mathcal{M}_s$ moyen et $\beta$ médian (volume).

In [ ]:
nv = sample(n; positive = true)
s = log.(nv ./ mean(nv))
σs = std(s; corrected = false); ms = mean(s)
Mach_mean = mean(Ms); β_med = median(β)
b_param = sqrt(expm1(σs^2)) / Mach_mean
b_mag = sqrt(expm1(σs^2) * (β_med + 1) / β_med) / Mach_mean

edges = collect(range(extrema(s)...; length = 81))
c, h = hist1d(s, edges); width = edges[2] - edges[1]
model = range(edges[1], edges[end]; length = 400)
fig = Figure(size = (960, 660))
ax = tax(fig[1, 1]; xlabel = L"s = \ln(n/\langle n\rangle)", ylabel = L"p(s)")
barplot!(ax, c, h ./ (length(s) * width); width = 0.94width, gap = 0, color = (COLORS[1], 0.35),
         strokecolor = COLORS[1], strokewidth = 0.7, label = L"\mathrm{measured}")
lines!(ax, model, @. exp(-(model + σs^2 / 2)^2 / (2σs^2)) / sqrt(2π * σs^2); color = COLORS[2], linewidth = 2.8,
       label = latexstring("\\mathrm{lognormal},\\ \\sigma_s = ", @sprintf("%.3f", σs)))
lines!(ax, model, @. exp(-(model - ms)^2 / (2σs^2)) / sqrt(2π * σs^2); color = COLORS[3], linewidth = 2.2,
       linestyle = :dash, label = latexstring("\\mathrm{Gaussian},\\ \\langle s\\rangle = ", @sprintf("%.3f", ms)))
vlines!(ax, [ms]; color = :gray50, linestyle = :dot,
        label = latexstring("b = ", @sprintf("%.3f", b_param), ",\\ b_B = ", @sprintf("%.3f", b_mag)))
axislegend(ax; position = :lt)
finish(fig, "density_pdf")

## 12. Bilan des phases thermiques

Fractions de volume (cellules) et de masse ($\sum n$) dans chaque phase.

In [ ]:
function phase_budget(n, T; stride = STRIDE)
    vol = zeros(length(PHASES)); mass = zeros(length(PHASES)); V = 0; Mtot = 0.0
    for i in 1:stride:length(n)
        (isfinite(n[i]) && isfinite(T[i]) && n[i] > 0) || continue
        V += 1; Mtot += n[i]
        p = phase_of(T[i]); p === nothing && continue
        vol[p] += 1; mass[p] += n[i]
    end
    return vol ./ V, mass ./ Mtot
end

fV, fM = phase_budget(n, T)
pos = collect(1.0:length(PHASES))
fig = Figure(size = (960, 620))
ax = tax(fig[1, 1]; xlabel = L"\mathrm{thermal\ phase}", ylabel = L"\mathrm{filling\ factor}",
         xticks = (pos, [latexstring("\\mathrm{", p.name, "}") for p in PHASES]))
barplot!(ax, pos .- 0.18, fV; width = 0.34, color = (COLORS[1], 0.85), label = L"\mathrm{volume}")
barplot!(ax, pos .+ 0.18, fM; width = 0.34, color = (COLORS[2], 0.85), label = L"\mathrm{mass}")
for (i, (a, b)) in enumerate(zip(fV, fM))
    text!(ax, pos[i] - 0.18, a + 0.015; text = latexstring(@sprintf("%.3f", a)), align = (:center, :bottom), fontsize = 14)
    text!(ax, pos[i] + 0.18, b + 0.015; text = latexstring(@sprintf("%.3f", b)), align = (:center, :bottom), fontsize = 14)
end
ylims!(ax, 0, 1.12)
axislegend(ax; position = :lt)
finish(fig, "phase_budget")

## 13. PDF de la densité de colonne

PDF de $N$ pour chaque ligne de visée en bins logarithmiques. La queue en loi de puissance
$p(N) \propto N^{-\alpha}$ est ajustée par maximum de vraisemblance (Clauset, Shalizi & Newman 2009) :
pour chaque seuil $N_{\min}$ candidat, $\alpha = 1 + n_{\mathrm{tail}}/\sum\ln(N/N_{\min})$, et on garde le
seuil qui minimise la distance de Kolmogorov-Smirnov (queue d'au moins 50 pixels et 2 % de la carte).

In [ ]:
function powerlaw_tail(values; candidates = 60, min_tail = 50, min_fraction = 0.02)
    x = sort(filter(v -> isfinite(v) && v > 0, values)); N = length(x)
    N >= min_tail || return (α = NaN, xmin = NaN)
    min_tail = max(min_tail, round(Int, min_fraction * N))
    suffix = reverse(cumsum(reverse(log.(x))))      # suffix[i] = Σ_{j≥i} ln x_j
    lo = max(1, searchsortedfirst(x, quantile(x, 0.30)))
    hi = max(lo, N - min_tail + 1)
    best = (α = NaN, xmin = NaN, ks = Inf)
    for start in lo:max(1, (hi - lo) ÷ candidates):hi
        ntail = N - start + 1; ntail >= min_tail || continue
        den = suffix[start] - ntail * log(x[start]); den > 0 || continue
        α = 1 + ntail / den
        ks = maximum(abs((i - start + 1) / ntail - (1 - (x[i] / x[start])^(1 - α))) for i in start:N)
        ks < best.ks && (best = (α = α, xmin = x[start], ks = ks))
    end
    return (α = best.α, xmin = best.xmin)
end

fig = Figure(size = (980, 660))
ax = tax(fig[1, 1]; xlog = true, ylog = true, xlabel = L"N\ [\mathrm{cm^{-2}}]", ylabel = L"p(N)")
for d in 1:3
    Nmap = vec(column_density(d)); Nmap = Nmap[Nmap .> 0]
    edges = 10 .^ range(log10(minimum(Nmap)), log10(maximum(Nmap)); length = 61)
    c, h = hist1d(Nmap, edges)
    pdf = h ./ (length(Nmap) .* diff(edges))
    good = pdf .> 0
    lines!(ax, c[good], pdf[good]; color = COLORS[d], linewidth = 2.6,
           label = latexstring("\\mathrm{LOS}\\ ", AXIS_NAMES[d]))
    fit = powerlaw_tail(Nmap)
    if isfinite(fit.α)
        tail = findall(x -> x >= fit.xmin, c)
        if !isempty(tail)
            a = tail[1]; ref = pdf[a] > 0 ? pdf[a] : maximum(pdf[good])
            lines!(ax, c[tail], ref .* (c[tail] ./ c[a]) .^ (-fit.α); color = COLORS[d], linestyle = :dashdot,
                   linewidth = 2, label = latexstring("\\alpha_", AXIS_NAMES[d], " = ", @sprintf("%.2f", fit.α)))
        end
    end
end
axislegend(ax; position = :lb)
finish(fig, "column_density_pdf")

## 14. Fonctions de structure

$S_p(\ell) = \langle|f(\mathbf{x}+\ell\,\hat{\mathbf{e}}) - f(\mathbf{x})|^p\rangle$, estimée par Monte-Carlo :
100 000 paires par décalage, axe et sens tirés au hasard, 18 décalages logarithmiques jusqu'à la demi-boîte.
L'exposant $\zeta_p$ est ajusté dans la plus longue fenêtre en loi de puissance entre l'échelle de dissipation
($2\pi/k_{\mathrm{diss}}$) et l'échelle d'injection ($2\pi/k_{\mathrm{inj}}$).

In [ ]:
# Tire une cellule et sa voisine à distance lag le long d'un axe aléatoire ; nothing si elle sort de la boîte
function random_pair(rng, sz, lag, periodic)
    ax = rand(rng, 1:3)
    dir = rand(rng, Bool) ? lag : -lag
    i, j, k = rand(rng, 1:sz[1]), rand(rng, 1:sz[2]), rand(rng, 1:sz[3])
    p = (i, j, k)[ax] + dir
    if periodic
        p = mod1(p, sz[ax])
    elseif !(1 <= p <= sz[ax])
        return nothing
    end
    target = ax == 1 ? (p, j, k) : ax == 2 ? (i, p, k) : (i, j, p)
    return (i, j, k), target, ax
end

function structure_functions(A; orders = (1, 2, 3), lags = log_lags(2, 18), samples = 100_000,
                             seed = 1234, periodic = PERIODIC)
    S = fill(NaN, length(lags), length(orders)); cnt = zeros(Int, length(lags))
    Threads.@threads for li in eachindex(lags)
        rng = MersenneTwister(seed + li)
        acc = zeros(length(orders)); c = 0
        for _ in 1:samples
            pair = random_pair(rng, size(A), lags[li], periodic)
            pair === nothing && continue
            a, b = A[pair[1]...], A[pair[2]...]
            (isfinite(a) && isfinite(b)) || continue
            δ = abs(Float64(b) - Float64(a))
            for (oi, o) in enumerate(orders)
                acc[oi] += δ^o
            end
            c += 1
        end
        cnt[li] = c
        c > 0 && (S[li, :] .= acc ./ c)
    end
    return S, cnt
end

SF_ORDERS = (1, 2, 3)
sf_lags = log_lags(2, 18)
ℓ = sf_lags .* DX[1]
ℓfit = (2π / KDIS, 2π / KINJ)

for (key, A, sym) in (("n", n, "n"), ("Vmag", Vmag, "|v|"))
    S, _ = structure_functions(A; orders = SF_ORDERS, lags = sf_lags)
    fig = Figure(size = (430 * length(SF_ORDERS), 520))
    for (col, p) in enumerate(SF_ORDERS)
        ax = tax(fig[1, col]; xlog = true, ylog = true, xlabel = L"\ell\ [\mathrm{pc}]",
                 ylabel = latexstring("S_{", p, "}^{", sym, "}(\\ell)"))
        shade_l!(ax, extrema(ℓ)...)
        good = findall(i -> isfinite(S[i, col]) && S[i, col] > 0, eachindex(ℓ))
        scatterlines!(ax, ℓ[good], S[good, col]; color = COLORS[col], linewidth = 2.6, markercolor = :white,
                      strokecolor = COLORS[col], strokewidth = 1.3, markersize = 7)
        rng_fit = best_powerlaw_range(ℓ, S[:, col], ℓfit)
        ζp, err = powerlaw_fit(ℓ, S[:, col], rng_fit)
        if isfinite(ζp)
            xs, ys = anchored_line(ℓ, S[:, col], ζp, rng_fit)
            lines!(ax, xs, ys; color = :black, linestyle = :dashdot, linewidth = 2.2,
                   label = latexstring("\\zeta_", p, " = ", @sprintf("%.2f", ζp), "\\pm", @sprintf("%.2f", err)))
            axislegend(ax; position = :lt)
        end
    end
    finish(fig, "structure_" * key)
end

### Auto-similarité étendue (ESS) par phase

Fonctions de structure de $|\delta\mathbf{v}|$ d'ordres 1 à 5, en ne gardant que les paires dont les deux
extrémités sont dans la même phase. $\zeta_p/\zeta_3$ est la pente de $\log S_p$ contre $\log S_3$ pour des
décalages entre 4 cellules et 1/4 de boîte (poids $\sqrt{N_{\mathrm{paires}}}$, erreur par bootstrap sur les
points). Références : Kolmogorov $p/3$, She-Lévêque $p/9 + 2[1 - (2/3)^{p/3}]$,
Boldyrev $p/9 + 1 - (1/3)^{p/3}$.

In [ ]:
function phase_velocity_sf(vx, vy, vz, T; orders = 1:5, lags = log_lags(2, 18), samples = 100_000,
                           seed = 1234, periodic = PERIODIC, phases = PHASES)
    S = zeros(length(lags), length(phases), length(orders)); cnt = zeros(Int, length(lags), length(phases))
    Threads.@threads for li in eachindex(lags)
        rng = MersenneTwister(seed + li)
        for _ in 1:samples
            pair = random_pair(rng, size(vx), lags[li], periodic)
            pair === nothing && continue
            I, J = pair[1], pair[2]
            p1, p2 = phase_of(T[I...]), phase_of(T[J...])
            (p1 !== nothing && p1 == p2) || continue
            du2 = (vx[J...] - vx[I...])^2 + (vy[J...] - vy[I...])^2 + (vz[J...] - vz[I...])^2
            isfinite(du2) || continue
            du = sqrt(Float64(du2))
            for (oi, o) in enumerate(orders)
                S[li, p1, oi] += du^o
            end
            cnt[li, p1] += 1
        end
    end
    for li in eachindex(lags), p in eachindex(phases)
        S[li, p, :] ./= max(cnt[li, p], 1)
        cnt[li, p] == 0 && (S[li, p, :] .= NaN)
    end
    return S, cnt
end

ESS_ORDERS = collect(1:5)
S_phase, cnt_phase = phase_velocity_sf(Vx, Vy, Vz, T; orders = ESS_ORDERS, lags = sf_lags)
lag_window = (DISSIPATION_CELLS, minimum(shape) / INJECTION_MODES)

function ess_exponents(S, cnt, lags; window = lag_window, replicates = 200, seed = 1234)
    i3 = findfirst(==(3), ESS_ORDERS)
    ζ = fill(NaN, size(S, 2)); err = fill(NaN, size(S, 2))
    for p in axes(S, 2)
        sel = findall(i -> window[1] <= lags[i] <= window[2] && cnt[i] > 0 && S[i, i3] > 0 && S[i, p] > 0, eachindex(lags))
        length(sel) >= 3 || continue
        x = log10.(S[sel, i3]); y = log10.(S[sel, p]); w = sqrt.(Float64.(cnt[sel]))
        xm = sum(w .* x) / sum(w); ym = sum(w .* y) / sum(w)
        ζ[p] = sum(w .* (x .- xm) .* (y .- ym)) / sum(w .* (x .- xm) .^ 2)
        rng = MersenneTwister(seed + p); boot = Float64[]
        for _ in 1:replicates
            r = rand(rng, eachindex(x), length(x))
            bx, by = x[r], y[r]; D = sum(abs2, bx .- mean(bx))
            D > 0 && push!(boot, sum((bx .- mean(bx)) .* (by .- mean(by))) / D)
        end
        length(boot) > 1 && (err[p] = std(boot; corrected = false))
    end
    return ζ, err
end

fig = Figure(size = (900, 620))
ax = tax(fig[1, 1]; xlabel = L"p", ylabel = L"\zeta_p/\zeta_3", xticks = ESS_ORDERS)
pp = range(1, 5; length = 100)
lines!(ax, pp, pp ./ 3; color = :black, linestyle = :dash, linewidth = 2, label = L"\mathrm{Kolmogorov}")
lines!(ax, pp, @. pp / 9 + 2 * (1 - (2 / 3)^(pp / 3)); color = :gray50, linestyle = :dot, linewidth = 2.2,
       label = L"\mathrm{She-Leveque}")
lines!(ax, pp, @. pp / 9 + 1 - (1 / 3)^(pp / 3); color = :black, linestyle = :dashdot, linewidth = 2,
       label = L"\mathrm{Boldyrev}")
for (ip, ph) in enumerate(PHASES)
    ζ, err = ess_exponents(S_phase[:, ip, :], cnt_phase[:, ip], sf_lags)
    ok = isfinite.(ζ)
    any(ok) || continue
    errorbars!(ax, ESS_ORDERS[ok], ζ[ok], replace(err[ok], NaN => 0.0); color = (COLORS[ip], 0.7), whiskerwidth = 7)
    scatterlines!(ax, ESS_ORDERS[ok], ζ[ok]; color = COLORS[ip], linewidth = 2.6, markercolor = :white,
                  strokecolor = COLORS[ip], strokewidth = 1.5, markersize = 10,
                  label = latexstring("\\mathrm{", ph.name, "}"))
end
axislegend(ax; position = :lt)
finish(fig, "phase_structure_ess")

## 15. Diagnostics par phase thermique

Fractions de volume et de masse avec une erreur par bootstrap sur $4^3$ sous-blocs ; dans chaque phase,
$\sigma_v = \sqrt{\langle|v|^2\rangle/3}$ et les médianes de $\mathcal{M}_s$, $\mathcal{M}_A$, $\beta$ et $|B|$.

In [ ]:
function phase_statistics(n, T, bx, by, bz, vx, vy, vz; stride = STRIDE, blocks = 4, replicates = 200,
                          seed = 1234, phases = PHASES)
    np = length(phases); sz = size(n); CI = CartesianIndices(n)
    cnt = zeros(Int, np); mass = zeros(np)
    bcnt = zeros(Int, blocks^3, np); bmass = zeros(blocks^3, np)
    v2sum = zeros(np)
    Ms = [Float64[] for _ in 1:np]; MA = [Float64[] for _ in 1:np]
    βs = [Float64[] for _ in 1:np]; Bm = [Float64[] for _ in 1:np]
    for i in 1:stride:length(n)
        ρ, t = n[i], T[i]
        (isfinite(ρ) && isfinite(t) && ρ > 0 && t > 0) || continue
        p = phase_of(t); p === nothing && continue
        vv = Float64(vx[i])^2 + Float64(vy[i])^2 + Float64(vz[i])^2
        bb = Float64(bx[i])^2 + Float64(by[i])^2 + Float64(bz[i])^2
        isfinite(vv + bb) || continue
        I = CI[i]
        blk = ntuple(d -> min(blocks, cld(I[d] * blocks, sz[d])), 3)
        b = blk[1] + blocks * (blk[2] - 1) + blocks^2 * (blk[3] - 1)
        cnt[p] += 1; mass[p] += ρ; bcnt[b, p] += 1; bmass[b, p] += ρ
        v2sum[p] += vv / 3
        cs = sqrt(GAMMA * KB * t / (MU * MH))
        push!(Ms[p], 1e5 * sqrt(vv) / cs)
        B = sqrt(bb); push!(Bm[p], B)
        if B > 0
            BG = 1e-6 * B
            push!(MA[p], 1e5 * sqrt(vv) / (BG / sqrt(4π * MU * MH * ρ)))
            push!(βs[p], 8π * ρ * KB * t / BG^2)
        end
    end
    rng = MersenneTwister(seed)
    bootV = [Float64[] for _ in 1:np]; bootM = [Float64[] for _ in 1:np]
    for _ in 1:replicates
        pick = rand(rng, 1:blocks^3, blocks^3)
        c = vec(sum(bcnt[pick, :]; dims = 1)); m = vec(sum(bmass[pick, :]; dims = 1))
        for p in 1:np
            push!(bootV[p], c[p] / max(sum(c), 1)); push!(bootM[p], m[p] / max(sum(m), eps()))
        end
    end
    med(v) = isempty(v) ? NaN : median(v)
    return [(name = phases[p].name, fV = cnt[p] / max(sum(cnt), 1), fM = mass[p] / max(sum(mass), eps()),
             eV = std(bootV[p]; corrected = false), eM = std(bootM[p]; corrected = false),
             σv = sqrt(v2sum[p] / max(cnt[p], 1)), Ms = med(Ms[p]), MA = med(MA[p]), β = med(βs[p]), B = med(Bm[p]))
            for p in 1:np]
end

stats = phase_statistics(n, T, Bx, By, Bz, Vx, Vy, Vz)
pos = collect(1.0:length(stats))
phase_ticks = (pos, [latexstring("\\mathrm{", s.name, "}") for s in stats])

fig = Figure(size = (920, 540))
ax = tax(fig[1, 1]; xlabel = L"\mathrm{thermal\ phase}", ylabel = L"\mathrm{fraction}", xticks = phase_ticks)
barplot!(ax, pos .- 0.18, [s.fV for s in stats]; width = 0.34, color = COLORS[1], label = L"f_V")
barplot!(ax, pos .+ 0.18, [s.fM for s in stats]; width = 0.34, color = COLORS[2], label = L"f_M")
errorbars!(ax, pos .- 0.18, [s.fV for s in stats], [s.eV for s in stats]; color = :black, whiskerwidth = 8)
errorbars!(ax, pos .+ 0.18, [s.fM for s in stats], [s.eM for s in stats]; color = :black, whiskerwidth = 8)
ylims!(ax, 0, nothing)
axislegend(ax; position = :lt)
finish(fig, "phase_fractions")

quantities = ((:σv, L"\sigma_v\ [\mathrm{km\,s^{-1}}]"), (:Ms, L"\mathcal{M}_s"), (:MA, L"\mathcal{M}_A"),
              (:β, L"\beta"), (:B, L"|B|\ [\mu\mathrm{G}]"))
fig = Figure(size = (1320, 760))
for (k, (field, lab)) in enumerate(quantities)
    ax = tax(fig[cld(k, 3), mod1(k, 3)]; ylog = true, xlabel = L"\mathrm{thermal\ phase}", ylabel = lab,
             xticks = phase_ticks)
    vals = [getfield(s, field) for s in stats]
    ok = isfinite.(vals) .& (vals .> 0)
    scatter!(ax, pos[ok], vals[ok]; markersize = 14, color = COLORS[mod1(k, 7)],
             marker = (:circle, :rect, :utriangle, :diamond, :cross)[k])
    xlims!(ax, 0.5, length(stats) + 0.5)
end
finish(fig, "phase_physical_diagnostics")

### Intermittence par phase

Incréments longitudinaux $\delta v_L = v_a(\mathbf{x}+\ell\hat{\mathbf{e}}_a) - v_a(\mathbf{x})$ (même phase aux deux
extrémités, 50 000 paires par décalage, 14 décalages). Asymétrie $\mathcal{S} = \langle\delta v_L^3\rangle/\langle\delta v_L^2\rangle^{3/2}$
et aplatissement $\mathcal{F} = \langle\delta v_L^4\rangle/\langle\delta v_L^2\rangle^2$ ($\mathcal{F} = 3$ pour une
gaussienne), puis PDF des incréments normalisés à trois échelles.

In [ ]:
function phase_increments(vx, vy, vz, T; lags = log_lags(2, 14), samples = 50_000, seed = 1234,
                          periodic = PERIODIC, phases = PHASES, keep = 5000)
    rng = MersenneTwister(seed)
    m = zeros(length(lags), length(phases), 3); cnt = zeros(Int, length(lags), length(phases))
    store = [Float64[] for _ in lags, _ in phases]
    comps = (vx, vy, vz)
    for (li, lag) in enumerate(lags), _ in 1:samples
        pair = random_pair(rng, size(vx), lag, periodic)
        pair === nothing && continue
        I, J, ax = pair
        p = phase_of(T[I...]); (p !== nothing && p == phase_of(T[J...])) || continue
        du = Float64(comps[ax][J...] - comps[ax][I...]); isfinite(du) || continue
        m[li, p, 1] += du^2; m[li, p, 2] += du^3; m[li, p, 3] += du^4; cnt[li, p] += 1
        length(store[li, p]) < keep && push!(store[li, p], du)
    end
    skew = fill(NaN, length(lags), length(phases)); flat = copy(skew)
    for li in eachindex(lags), p in eachindex(phases)
        c = cnt[li, p]; c > 2 || continue
        m2 = m[li, p, 1] / c; m2 > 0 || continue
        skew[li, p] = (m[li, p, 2] / c) / m2^1.5; flat[li, p] = (m[li, p, 3] / c) / m2^2
    end
    return skew, flat, store
end

inc_lags = log_lags(2, 14)
skew, flat, store = phase_increments(Vx, Vy, Vz, T; lags = inc_lags)
ℓinc = inc_lags .* DX[1]

fig = Figure(size = (1000, 480))
for (col, (vals, lab)) in enumerate(((skew, L"\mathcal{S}"), (flat, L"\mathcal{F}")))
    ax = tax(fig[1, col]; xlog = true, xlabel = L"\ell\ [\mathrm{pc}]", ylabel = lab)
    for (p, ph) in enumerate(PHASES)
        lines!(ax, ℓinc, vals[:, p]; color = COLORS[p], linestyle = (:solid, :dash, :dot)[mod1(p, 3)],
               linewidth = 2.5, label = latexstring("\\mathrm{", ph.name, "}"))
    end
    col == 1 && axislegend(ax; position = :lt)
end
finish(fig, "phase_intermittency")

chosen = unique(round.(Int, range(1, length(inc_lags); length = min(3, length(inc_lags)))))
fig = Figure(size = (440 * length(chosen), 480))
pdf_edges = collect(range(-6, 6; length = 61)); pdf_w = pdf_edges[2] - pdf_edges[1]
for (col, li) in enumerate(chosen)
    ax = tax(fig[1, col]; ylog = true, xlabel = L"\delta v_L/\sigma_{\delta v}", ylabel = L"\mathrm{PDF}",
             title = latexstring("\\ell = ", @sprintf("%.2g", ℓinc[li]), "\\ \\mathrm{pc}"))
    for (p, ph) in enumerate(PHASES)
        v = store[li, p]; length(v) > 10 || continue
        z = (v .- mean(v)) ./ std(v)
        c, h = hist1d(z, pdf_edges)
        h ./= max(sum(h) * pdf_w, eps())
        good = h .> 0
        lines!(ax, c[good], h[good]; color = COLORS[p], linestyle = (:solid, :dash, :dot)[mod1(p, 3)],
               linewidth = 2.4, label = latexstring("\\mathrm{", ph.name, "}"))
    end
    lines!(ax, pdf_edges, @. exp(-pdf_edges^2 / 2) / sqrt(2π); color = :gray50, linewidth = 1.5)
    col == 1 && axislegend(ax; position = :lt)
end
finish(fig, "phase_increment_pdfs")

## 16. Topologie des régions denses

Ensembles d'excursion $\{n \ge n_{\mathrm{th}}\}$ pour des seuils aux quantiles 50, 70, 80, 90, 95 et 99 %.
Les quatre fonctionnelles de Minkowski par unité de volume : fraction de volume, densité de surface (faces de
cellules séparant l'intérieur de l'extérieur), largeur moyenne $V_1 = n_1 - 2n_2 + 3n_3$ et caractéristique
d'Euler $\chi = n_0 - n_1 + n_2 - n_3$, où $n_0, n_1, n_2, n_3$ comptent sommets, arêtes, faces et cubes du complexe
cubique. Puis le nombre de composantes connexes (6-connexité) et la fraction du volume dans la plus grande.

In [ ]:
function excursion(field, threshold; periodic = PERIODIC, dxs = DX)
    act = falses(size(field))
    for i in eachindex(field)
        act[i] = isfinite(field[i]) && field[i] >= threshold
    end
    sz = size(act); V = prod(sz .* dxs)
    nact = count(act)
    nact == 0 && return (fV = 0.0, S = 0.0, V1 = 0.0, χ = 0.0, ncomp = 0, largest = 0.0)
    face = (dxs[2] * dxs[3], dxs[1] * dxs[3], dxs[1] * dxs[2])

    # surface : faces entre une cellule active et une inactive
    S = 0.0
    for k in 1:sz[3], j in 1:sz[2], i in 1:sz[1], d in 1:3
        I = (i, j, k); nxt = I[d] + 1
        if nxt > sz[d]
            if periodic
                nxt = 1
            else
                act[i, j, k] && (S += face[d])
                continue
            end
        end
        J = d == 1 ? (nxt, j, k) : d == 2 ? (i, nxt, k) : (i, j, nxt)
        act[i, j, k] != act[J...] && (S += face[d])
        !periodic && act[i, j, k] && I[d] == 1 && (S += face[d])
    end

    # complexe cubique : sommets, arêtes et faces touchés par une cellule active
    ext = periodic ? sz : sz .+ 1
    wrap(x, d) = periodic ? mod1(x, sz[d]) : x
    vert = falses(ext); edg = ntuple(_ -> falses(ext), 3); fac = ntuple(_ -> falses(ext), 3)
    for k in 1:sz[3], j in 1:sz[2], i in 1:sz[1]
        act[i, j, k] || continue
        for c in 0:1, b in 0:1, a in 0:1
            vert[wrap(i + a, 1), wrap(j + b, 2), wrap(k + c, 3)] = true
        end
        for c in 0:1, b in 0:1
            edg[1][i, wrap(j + b, 2), wrap(k + c, 3)] = true
            edg[2][wrap(i + b, 1), j, wrap(k + c, 3)] = true
            edg[3][wrap(i + b, 1), wrap(j + c, 2), k] = true
        end
        for a in 0:1
            fac[1][wrap(i + a, 1), j, k] = true
            fac[2][i, wrap(j + a, 2), k] = true
            fac[3][i, j, wrap(k + a, 3)] = true
        end
    end
    n0 = count(vert); n1 = sum(count, edg); n2 = sum(count, fac); n3 = nact
    χ = n0 - n1 + n2 - n3
    V1 = dxs[1] * (n1 - 2n2 + 3n3)

    # composantes connexes par parcours en largeur
    seen = falses(sz); queue = Int[]; LI = LinearIndices(act); CI = CartesianIndices(act)
    ncomp = 0; largest = 0
    for seed in eachindex(act)
        (act[seed] && !seen[seed]) || continue
        ncomp += 1; empty!(queue); push!(queue, seed); seen[seed] = true; head = 1
        while head <= length(queue)
            I = CI[queue[head]]; head += 1
            for d in 1:3, off in (-1, 1)
                c = I[d] + off
                if periodic
                    c = mod1(c, sz[d])
                elseif !(1 <= c <= sz[d])
                    continue
                end
                J = d == 1 ? LI[c, I[2], I[3]] : d == 2 ? LI[I[1], c, I[3]] : LI[I[1], I[2], c]
                if act[J] && !seen[J]
                    seen[J] = true; push!(queue, J)
                end
            end
        end
        largest = max(largest, length(queue))
    end
    return (fV = nact / length(act), S = S / V, V1 = V1 / V, χ = χ / V, ncomp = ncomp, largest = largest / nact)
end

thresholds = quantile(sample(n; positive = true), [0.50, 0.70, 0.80, 0.90, 0.95, 0.99])
topo = Vector{Any}(undef, length(thresholds))
Threads.@threads for i in eachindex(thresholds)
    topo[i] = excursion(n, thresholds[i])
end
xlab = L"n_{\mathrm{th}}\ [\mathrm{cm^{-3}}]"

fig = Figure(size = (1400, 780))
for (k, (field, lab, col)) in enumerate(((:fV, L"V_0/V\ (\mathrm{volume\ fraction})", COLORS[1]),
                                         (:S,  L"S/V\ [\mathrm{pc^{-1}}]", COLORS[2]),
                                         (:V1, L"V_1/V\ [\mathrm{pc^{-2}}]", COLORS[3]),
                                         (:χ,  L"\chi/V\ [\mathrm{pc^{-3}}]", COLORS[4])))
    ax = tax(fig[fldmod1(k, 2)...]; xlog = true, xlabel = xlab, ylabel = lab)
    field == :χ && hlines!(ax, [0.0]; color = :gray50, linestyle = :dash)
    scatterlines!(ax, thresholds, [getfield(t, field) for t in topo]; color = col, linewidth = 2.7,
                  markercolor = :white, strokecolor = col, strokewidth = 1.3, markersize = 7)
end
finish(fig, "minkowski_dense_gas")

fig = Figure(size = (1150, 520))
ax1 = tax(fig[1, 1]; xlog = true, xlabel = xlab, ylabel = L"\mathrm{fraction}")
lines!(ax1, thresholds, [t.fV for t in topo]; color = COLORS[1], linewidth = 2.7, label = L"\mathrm{volume}")
lines!(ax1, thresholds, [t.largest for t in topo]; color = COLORS[2], linewidth = 2.5, linestyle = :dash,
       label = L"\mathrm{largest\ component}")
axislegend(ax1; position = :rt)
ax2 = tax(fig[1, 2]; xlog = true, xlabel = xlab, ylabel = L"N_{\mathrm{components}}")
lines!(ax2, thresholds, [t.ncomp for t in topo]; color = COLORS[3], linewidth = 2.7)
finish(fig, "topology_dense_gas")

## 17. Spectre anisotrope de la densité

Puissance moyenne par mode dans le plan $(k_\parallel, k_\perp)$, avec $k_\parallel = |\mathbf{k}\cdot\hat{\mathbf{b}}|$
et $\hat{\mathbf{b}}$ la direction du champ moyen $\langle\mathbf{B}\rangle$. Des contours allongés selon $k_\perp$
indiquent des structures étirées le long du champ.

In [ ]:
bhat = normalize([mean(sample(Bx)), mean(sample(By)), mean(sample(Bz))])

function anisotropic_power(A, bhat; bins = 32)
    F = fourier(A); kx, ky, kz = wavenumbers(size(A))
    kmin, kmax = KMIN, KMAX
    edges = collect(range(0, kmax; length = bins + 1))
    P = zeros(bins, bins); M = zeros(Int, bins, bins)
    for k in axes(F, 3), j in axes(F, 2), i in axes(F, 1)
        r = sqrt(kx[i]^2 + ky[j]^2 + kz[k]^2); kmin <= r <= kmax || continue
        kpar = abs(kx[i] * bhat[1] + ky[j] * bhat[2] + kz[k] * bhat[3])
        kperp = sqrt(max(r^2 - kpar^2, 0))
        a = clamp(searchsortedlast(edges, kpar), 1, bins); b = clamp(searchsortedlast(edges, kperp), 1, bins)
        w = (i == 1 || (iseven(size(A, 1)) && i == size(F, 1))) ? 1 : 2
        P[a, b] += w * abs2(F[i, j, k]); M[a, b] += w
    end
    P = [M[i] > 0 ? P[i] / M[i] : NaN for i in eachindex(P)]
    c = (edges[1:end-1] .+ edges[2:end]) ./ 2
    return c, reshape(P, bins, bins)
end

kc, P2 = anisotropic_power(n, bhat)
L2 = map(x -> isfinite(x) && x > 0 ? log10(x) : NaN, P2)
fig = Figure(size = (900, 700))
ax = tax(fig[1, 1]; aspect = DataAspect(), xlabel = L"k_\parallel\ [\mathrm{pc^{-1}}]", ylabel = L"k_\perp\ [\mathrm{pc^{-1}}]",
         title = latexstring("\\langle B\\rangle/|\\langle B\\rangle| = (", join([@sprintf("%.2f", x) for x in bhat], ",\\,"), ")"))
hm = heatmap!(ax, kc, kc, L2; colormap = :magma, nan_color = :white)
texcolorbar(fig[1, 2], hm; label = L"\log_{10}\,P_n(k_\parallel, k_\perp)")
finish(fig, "anisotropic_spectrum_density")

## 18. Fonctions de structure directionnelles de la vitesse

Pour chaque axe $a$ et chaque décalage (12 décalages jusqu'au quart de boîte, toutes les cellules) :
fonction longitudinale $\langle|\delta v_a|^p\rangle$ (traits pleins) et transverse $\langle|\delta\mathbf{v}_\perp|^p\rangle$
(tirets). Même chose pour $n^{1/3}\mathbf{v}$, dont la fonction d'ordre 3 est l'analogue compressible du flux d'énergie.

In [ ]:
function directional_sf(cx, cy, cz; orders = 1:4, lags = log_lags(4, 12), weight = nothing, wpow = 1 / 3,
                        periodic = PERIODIC)
    comps = (cx, cy, cz); sz = size(cx)
    L = zeros(3, length(lags), length(orders)); Tr = zeros(3, length(lags), length(orders))
    tasks = [(a, li) for a in 1:3 for li in eachindex(lags)]
    Threads.@threads for ti in eachindex(tasks)
        a, li = tasks[ti]
        lag = lags[li]; accL = zeros(length(orders)); accT = zeros(length(orders)); c = 0
        for k in 1:sz[3], j in 1:sz[2], i in 1:sz[1]
            I = (i, j, k); t = I[a] + lag
            if t > sz[a]
                periodic || continue
                t = mod1(t, sz[a])
            end
            J = a == 1 ? (t, j, k) : a == 2 ? (i, t, k) : (i, j, t)
            w1 = weight === nothing ? 1.0 : max(Float64(weight[I...]), 0.0)^wpow
            w2 = weight === nothing ? 1.0 : max(Float64(weight[J...]), 0.0)^wpow
            δ = ntuple(d -> w2 * Float64(comps[d][J...]) - w1 * Float64(comps[d][I...]), 3)
            all(isfinite, δ) || continue
            dl = abs(δ[a]); dt = sqrt(sum(δ[d]^2 for d in 1:3 if d != a))
            for (oi, o) in enumerate(orders)
                accL[oi] += dl^o; accT[oi] += dt^o
            end
            c += 1
        end
        c > 0 && (L[a, li, :] .= accL ./ c; Tr[a, li, :] .= accT ./ c)
    end
    return L, Tr
end

dir_lags = log_lags(4, 12)
for (name, weight, orders, sym) in (("velocity", nothing, 1:4, "v"), ("density_weighted_velocity", n, 1:3, "n^{1/3}v"))
    L, Tr = directional_sf(Vx, Vy, Vz; orders = orders, lags = dir_lags, weight = weight)
    fig = Figure(size = (430 * length(orders), 540))
    for (col, p) in enumerate(orders)
        ax = tax(fig[1, col]; xlog = true, ylog = true, xlabel = L"\ell\ [\mathrm{pc}]",
                 ylabel = latexstring("S_{", p, "}^{", sym, "}(\\ell)"))
        for a in 1:3
            ℓa = dir_lags .* DX[a]
            lines!(ax, ℓa, L[a, :, col]; color = COLORS[a], linewidth = 2.6,
                   label = latexstring("\\mathrm{L},\\ ", AXIS_NAMES[a]))
            lines!(ax, ℓa, Tr[a, :, col]; color = COLORS[a], linewidth = 2.2, linestyle = :dash,
                   label = latexstring("\\mathrm{T},\\ ", AXIS_NAMES[a]))
        end
        col == 1 && axislegend(ax; position = :lt)
    end
    finish(fig, "directional_structure_" * name)
end